In [ ]:
# Load or reload R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    try:
        %reload_ext rpy2.ipython
    except Exception as e2:
        print("Note on rpy2 initialization:", e2)

# Transpiling 2-Tier Hierarchical LightGBM Pipeline to Native C (`models/transpile_soft_pipeline_to_c.ipynb`)

This notebook exports and transpiles the **Optimized 2-Tier Hierarchical LightGBM Soft Triage Pipeline** into zero-dependency **Native C Code** using `m2cgen` (Model 2 Code Generator) and tests the compiled C implementation against the original Python/R pipeline on the **Holdout Test Set**:

### 2-Tier Hierarchical LightGBM Architecture (< 250 KB C Code)
1. **Layer 1 LightGBM (ESI 1 Detector - Majority Undersampling)**: Binary Classifier trained with **majority class random undersampling** (`n_estimators = 100`).
2. **Layer 2 LightGBM (Direct 4-Class Specialist - ESI 5 Oversampling)**: Multiclass Classifier trained with **random bootstrap oversampling of ESI 5** (`n_estimators = 100`).
3. **Master C Function (`predict_soft_pipeline`)**:
   - $P(\text{ESI 1}) = P_{L1}(\text{ESI 1})$
   - $P(\text{ESI 2}) = (1 - P_{L1}(\text{ESI 1})) \times P_{L2}(\text{ESI 2})$
   - $P(\text{ESI 3}) = (1 - P_{L1}(\text{ESI 1})) \times P_{L2}(\text{ESI 3})$
   - $P(\text{ESI 4}) = (1 - P_{L1}(\text{ESI 1})) \times P_{L2}(\text{ESI 4})$
   - $P(\text{ESI 5}) = (1 - P_{L1}(\text{ESI 1})) \times P_{L2}(\text{ESI 5})$

### Transpilation & Integration Steps
1. **Load Data & Prepare 35 Features in R**: Reads dataset and constructs 35 features.
2. **Fit 2 LightGBM Sub-Models in Python**: Trains Layer 1 binary detector (undersampled) and Layer 2 4-class specialist (ESI 5 oversampled).
3. **Transpile Sub-Models to C via `m2cgen`**: Exports to `score_layer1_lgb1` and `score_layer2_lgb2_4class` with macro scopes.
4. **Compile & Test Shared C Library (`deploy/soft_triage_pipeline.so`)**: Compiles with `gcc -O3 -shared -fPIC` (< 0.3 seconds).
5. **Equivalence Verification & Benchmark Report**: Evaluates **Recall**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC**.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Data & Prepare 35 Predictor Features in R
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(pROC)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last"); p_max    <- get_vec("pulse_max"); p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last");   s_max    <- get_vec("sbp_max");   s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last");  o2_max   <- get_vec("spo2_max");  o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last");   r_max    <- get_vec("resp_max");   r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
df_full <- data.frame(
  age = raw_df$age, gender = gender_vec, cc_breathingdifficulty = cc_bd_vec,
  triage_vital_hr = t_hr, triage_vital_sbp = t_sbp, triage_vital_rr = t_rr, triage_vital_o2 = t_o2,
  pulse_last = p_last, resp_last = r_last, spo2_last = o2_last, sbp_last = s_last,
  pulse_min = p_min, resp_min = r_min, spo2_min = o2_min, sbp_min = s_min,
  pulse_max = p_max, resp_max = r_max, spo2_max = o2_max, sbp_max = s_max,
  hr_mean_to_last = t_hr - p_last, sbp_mean_to_last = t_sbp - s_last, spo2_mean_to_last = t_o2 - o2_last, rr_mean_to_last = t_rr - r_last,
  hr_range = p_max - p_min, rr_range = r_max - r_min, spo2_range = o2_max - o2_min, sbp_range = s_max - s_min,
  hr_last_to_min = p_last - p_min, rr_last_to_min = r_last - r_min, spo2_last_to_min = o2_last - o2_min, sbp_last_to_min = s_last - s_min,
  hr_last_to_max = p_last - p_max, rr_last_to_max = r_last - r_max, spo2_last_to_max = o2_last - o2_max, sbp_last_to_max = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_full <- na.omit(df_full)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Export partitions to R global environment
train_py <<- train_df
val_py   <<- val_df
test_py  <<- test_df
cat(sprintf("Partitions Prepared: Train=%d, Val=%d, Test=%d\n", nrow(train_df), nrow(val_df), nrow(test_df)))

In [ ]:
# ---------------------------------------------------------
# Step 2: Retrieve R Dataframes in Python & Fit 2 LightGBM Sub-Models
# ---------------------------------------------------------
import os
import numpy as np
import pandas as pd
from rpy2.robjects import r
import rpy2.robjects.pandas2ri as pandas2ri
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import m2cgen as m2c
# Retrieve dataframes directly from rpy2 environment
try:
    pandas2ri.activate()
    train_df = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['train_py']))
    val_df   = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['val_py']))
    test_df  = pd.DataFrame(pandas2ri.rpy2py_dataframe(r['test_py']))
except Exception:
    train_df = pd.DataFrame(r['train_py'])
    val_df   = pd.DataFrame(r['val_py'])
    test_df  = pd.DataFrame(r['test_py'])
feature_cols = [c for c in train_df.columns if c != 'target_col']
binary_cols  = ['gender', 'cc_breathingdifficulty']
cont_cols    = [c for c in feature_cols if c not in binary_cols]
# Standard scale continuous features
scaler = StandardScaler()
X_train_cont = scaler.fit_transform(train_df[cont_cols])
X_val_cont   = scaler.transform(val_df[cont_cols])
X_test_cont  = scaler.transform(test_df[cont_cols])
X_train = np.hstack([X_train_cont, train_df[binary_cols].values])
X_val   = np.hstack([X_val_cont,   val_df[binary_cols].values])
X_test  = np.hstack([X_test_cont,  test_df[binary_cols].values])
# Raw Targets
y_raw_tr = train_df['target_col'].astype(str).values
y_raw_ts = test_df['target_col'].astype(str).values
# ---------------------------------------------------------
# 1. Layer 1 LightGBM (ESI 1 Detector - Majority Undersampling)
# ---------------------------------------------------------
y_l1_tr = (y_raw_tr == '1').astype(int)
esi1_idx = np.where(y_l1_tr == 1)[0]
non1_idx = np.where(y_l1_tr == 0)[0]
np.random.seed(42)
sampled_non1_idx = np.random.choice(non1_idx, size=len(esi1_idx) * 2, replace=False)
l1_tr_idx = np.concatenate([esi1_idx, sampled_non1_idx])
X_tr_l1 = X_train[l1_tr_idx]
y_tr_l1_sub = y_l1_tr[l1_tr_idx]
print(f"Training Layer 1 LightGBM (ESI 1 Undersampled, {X_tr_l1.shape[0]} samples)...")
lgb_l1 = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    random_state=42,
    verbose=-1
)
lgb_l1.fit(X_tr_l1, y_tr_l1_sub)
# ---------------------------------------------------------
# 2. Layer 2 LightGBM (Direct 4-Class Specialist - ESI 5 Oversampling)
# ---------------------------------------------------------
no_esi1_idx = (y_raw_tr != '1')
X_tr_l2_raw = X_train[no_esi1_idx]
y_l2_raw = y_raw_tr[no_esi1_idx]
idx_2 = np.where(y_l2_raw == '2')[0]
idx_3 = np.where(y_l2_raw == '3')[0]
idx_4 = np.where(y_l2_raw == '4')[0]
idx_5 = np.where(y_l2_raw == '5')[0]
target_esi5_cnt = len(idx_4)
np.random.seed(42)
idx_5_over = np.random.choice(idx_5, size=target_esi5_cnt, replace=True)
l2_resampled_idx = np.concatenate([idx_2, idx_3, idx_4, idx_5_over])
X_tr_l2 = X_tr_l2_raw[l2_resampled_idx]
y_tr_l2_4cls = y_l2_raw[l2_resampled_idx].astype(int) - 2
print(f"Training Layer 2 Direct 4-Class LightGBM (ESI 5 Oversampled, {X_tr_l2.shape[0]} samples)...")
lgb_l2 = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    random_state=42,
    verbose=-1
)
lgb_l2.fit(X_tr_l2, y_tr_l2_4cls)
# Transpile Sub-Models to C Code via m2cgen
print("\nTranspiling 2 sub-models to C with m2cgen...")
code_lgb1  = m2c.export_to_c(lgb_l1, function_name="score_layer1_lgb1")
code_lgb2  = m2c.export_to_c(lgb_l2, function_name="score_layer2_lgb2_4class")
# Wrap sub-models in C preprocessor macro scopes
block_lgb1 = f"#define sigmoid lgb1_sigmoid\n{code_lgb1}\n#undef sigmoid"
block_lgb2 = f"#define softmax lgb2_softmax\n#define exp lgb2_exp\n{code_lgb2}\n#undef softmax\n#undef exp"
print("C Code transpilation & macro wrapping complete!")

In [ ]:
# ---------------------------------------------------------
# Step 3: Assemble Full C Source File & Soft Probabilistic Pipeline Wrapper
# ---------------------------------------------------------
deploy_dir = "../deploy"
if not os.path.exists(deploy_dir):
    deploy_dir = "deploy"
os.makedirs(deploy_dir, exist_ok=True)
c_file_path = os.path.abspath(os.path.join(deploy_dir, "soft_triage_pipeline.c"))
so_file_path = os.path.abspath(os.path.join(deploy_dir, "soft_triage_pipeline.so"))
c_header = """
/* =========================================================================
   2-Tier Hierarchical LightGBM Soft Triage Pipeline Master C File
   ========================================================================= */
#include <math.h>
#include <stddef.h>
"""
c_pipeline_wrapper = """
/* =========================================================================
   Soft Probabilistic Triage Pipeline Master C Function
   ========================================================================= */
void score_layer1_lgb1(double * input, double * output);
void score_layer2_lgb2_4class(double * input, double * output);
void predict_soft_pipeline(double * input, double * output_probs, int * pred_class) {
    // 1. Layer 1 LightGBM (ESI 1 Detector)
    double lgb1_probs[2];
    score_layer1_lgb1(input, lgb1_probs);
    double p_esi1 = lgb1_probs[1]; // P(ESI 1)
    double p_non1 = lgb1_probs[0]; // P(Non-ESI 1) = 1 - P(ESI 1)
    
    // 2. Layer 2 Direct 4-Class LightGBM (ESI 2, 3, 4, 5 Specialist)
    double lgb2_probs[4];
    score_layer2_lgb2_4class(input, lgb2_probs);
    
    // 3. Soft Joint Probability Multiplication
    output_probs[0] = p_esi1;                 // ESI 1
    output_probs[1] = p_non1 * lgb2_probs[0]; // ESI 2
    output_probs[2] = p_non1 * lgb2_probs[1]; // ESI 3
    output_probs[3] = p_non1 * lgb2_probs[2]; // ESI 4
    output_probs[4] = p_non1 * lgb2_probs[3]; // ESI 5
    
    // 4. Argmax Predicted Class (1..5)
    int best_cls = 1;
    double max_p = output_probs[0];
    for (int i = 1; i < 5; i++) {
        if (output_probs[i] > max_p) {
            max_p = output_probs[i];
            best_cls = i + 1;
        }
    }
    *pred_class = best_cls;
}
"""
full_c_code = c_header + "\n\n" + block_lgb1 + "\n\n" + block_lgb2 + "\n\n" + c_pipeline_wrapper
with open(c_file_path, "w") as f:
    f.write(full_c_code)
size_kb = os.path.getsize(c_file_path) / 1024
print(f"Master C Pipeline code written to: {c_file_path} ({size_kb:.2f} KB)")

In [ ]:
# ---------------------------------------------------------
# Step 4: Fast Compile C Code into Shared Library (.so)
# ---------------------------------------------------------
import subprocess
compile_cmd = f"gcc -O3 -shared -fPIC -lm '{c_file_path}' -o '{so_file_path}'"
print(f"Executing: {compile_cmd}")
res = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
if res.returncode == 0:
    print(f"SUCCESS: Dynamic C Shared Library compiled -> {so_file_path}")
else:
    raise RuntimeError(f"GCC Compilation Failed:\n{res.stderr}")

In [ ]:
# ---------------------------------------------------------
# Step 5: Run C Shared Library Inference & Compare with Original Python Output
# ---------------------------------------------------------
import ctypes
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
print(f"Loading C shared library from: {so_file_path}")
c_lib = ctypes.CDLL(so_file_path)
# Define function signature for predict_soft_pipeline
# void predict_soft_pipeline(double * input, double * output_probs, int * pred_class)
c_lib.predict_soft_pipeline.argtypes = [
    ctypes.POINTER(ctypes.c_double),
    ctypes.POINTER(ctypes.c_double),
    ctypes.POINTER(ctypes.c_int)
]
c_lib.predict_soft_pipeline.restype = None
# 1. Python Soft Pipeline Inference
p_l1_esi1_py = lgb_l1.predict_proba(X_test)[:, 1]
p_l2_mat_py  = lgb_l2.predict_proba(X_test)
p_non1_py = 1.0 - p_l1_esi1_py
N = X_test.shape[0]
probs_py = np.zeros((N, 5))
probs_py[:, 0] = p_l1_esi1_py                             # ESI 1
probs_py[:, 1] = p_non1_py * p_l2_mat_py[:, 0]            # ESI 2
probs_py[:, 2] = p_non1_py * p_l2_mat_py[:, 1]            # ESI 3
probs_py[:, 3] = p_non1_py * p_l2_mat_py[:, 2]            # ESI 4
probs_py[:, 4] = p_non1_py * p_l2_mat_py[:, 3]            # ESI 5
preds_py = np.argmax(probs_py, axis=1) + 1
# 2. C Shared Library Soft Pipeline Inference
probs_c = np.zeros((N, 5), dtype=np.float64)
preds_c = np.zeros(N, dtype=np.int32)
for i in range(N):
    x_sample = X_test[i, :].astype(np.float64)
    x_ptr = x_sample.ctypes.data_as(ctypes.POINTER(ctypes.c_double))
    
    out_p = (ctypes.c_double * 5)()
    out_cls = ctypes.c_int()
    
    c_lib.predict_soft_pipeline(x_ptr, out_p, ctypes.byref(out_cls))
    
    probs_c[i, :] = np.array([out_p[k] for k in range(5)])
    preds_c[i] = out_cls.value
# 3. Check Exact Numerical Equality
max_prob_diff = np.max(np.abs(probs_py - probs_c))
mismatched_preds = np.sum(preds_py != preds_c)
print(f"============================================================")
print(f"   C TRANSPILATION VERIFICATION & CHANGE AUDIT REPORT")
print(f"============================================================")
print(f"  Holdout Test Set Size               : {N} samples")
print(f"  Max Prob Probability Difference     : {max_prob_diff:.10e}")
print(f"  Class Prediction Mismatches (C vs PY): {mismatched_preds} / {N} ({(mismatched_preds/N)*100:.2f}%)")
print(f"============================================================\n")
if max_prob_diff < 1e-6 and mismatched_preds == 0:
    print("VERDICT: The transpiled C implementation is 100% IDENTICAL to the Python/R soft pipeline. No prediction changes occurred!")
else:
    print("VERDICT: Numerical variations detected between native C execution and original python model.")

In [ ]:
# ---------------------------------------------------------
# Step 6: Benchmark Transpiled C Model (Recall, Specificity, BalAcc, ROC-AUC ONLY)
# ---------------------------------------------------------
y_true = test_df['target_col'].astype(int).values
def compute_benchmark_metrics(y_true, y_pred, probs):
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5])
    
    rec_list, spec_list, bal_acc_list, auc_list = [], [], [], []
    for i in range(5):
        cls = i + 1
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - (tp + fn + fp)
        
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal_acc = (rec + spec) / 2.0
        
        y_bin = (y_true == cls).astype(int)
        try:
            auc = roc_auc_score(y_bin, probs[:, i])
        except Exception:
            auc = np.nan
            
        rec_list.append(rec)
        spec_list.append(spec)
        bal_acc_list.append(bal_acc)
        auc_list.append(auc)
    macro_rec  = np.mean(rec_list)
    macro_spec = np.mean(spec_list)
    macro_bal  = np.mean(bal_acc_list)
    macro_auc  = np.nanmean(auc_list)
    
    return macro_rec, macro_spec, macro_bal, macro_auc, cm
rec_c, spec_c, bal_c, auc_c, cm_c = compute_benchmark_metrics(y_true, preds_c, probs_c)
rec_py, spec_py, bal_py, auc_py, _ = compute_benchmark_metrics(y_true, preds_py, probs_py)
print("============================================================")
print("   TRANSPILED 2-TIER LIGHTGBM C MODEL HOLDOUT BENCHMARK")
print("============================================================")
print(f"  Macro Recall (Sensitivity) : {rec_c:.4f}  (Python: {rec_py:.4f})")
print(f"  Macro Specificity          : {spec_c:.4f}  (Python: {spec_py:.4f})")
print(f"  Macro Balanced Accuracy    : {bal_c:.4f}  (Python: {bal_py:.4f})")
print(f"  Macro ROC-AUC              : {auc_c:.4f}  (Python: {auc_py:.4f})")
print("============================================================\n")
print("Confusion Matrix (Native C Implementation):")
print(cm_c)
# Save C Transpilation Test Report
reports_dir = "../reports"
if not os.path.exists(reports_dir):
    reports_dir = "reports"
os.makedirs(reports_dir, exist_ok=True)
c_report_df = pd.DataFrame({
    'Implementation': ['Native_C_Transpiled', 'Original_Python'],
    'Macro_Recall': [round(rec_c, 4), round(rec_py, 4)],
    'Macro_Specificity': [round(spec_c, 4), round(spec_py, 4)],
    'Macro_Balanced_Accuracy': [round(bal_c, 4), round(bal_py, 4)],
    'Macro_ROC_AUC': [round(auc_c, 4), round(auc_py, 4)]
})
c_report_df.to_csv(os.path.join(reports_dir, "c_transpilation_test_report.csv"), index=False)
print(f"\nC Transpilation Test Report written to: {os.path.join(reports_dir, 'c_transpilation_test_report.csv')}")